In [24]:
# ============ RECALL Week 4 Day 2 -WARMUP ============
# 1. Bagging vs single decision tree:
  # Bagging works on Averaging n random tress on each split.
  #Bagging trains N trees independently on bootstrap samples (random samples with replacement) and aggregates predictions via majority vote.
  #A single decision tree trains on the full dataset once, making it high variance and prone to overfitting.

# 2. Why dropping Pclass hurt:
  # Feature_selection shows Pclass as the last option needed. But last choice does not mean not needed at all and thus reduces the model efficiency.
  #Low feature importance indicates a feature contributes less relative to others, not that it contributes zero.
  #Pclass retained unique survival signal not fully captured by the correlated Fare feature, removing it caused information loss and reduced CV score from 0.814 to 0.789.

# 3. What n_estimators controls:
  #no of trees to be used in random forest
  #The number of trees in the Random Forest. More trees reduces variance up to a point, after which returns diminish ,score stabilised beyond 100 trees
  #in our experiment with less than 1% variation across [10, 50, 100, 200, 500]

# =======================================

1. Gradient Boosting: how each tree corrects the previous tree's errors
2. XGBoost: install, train on Titanic — beat your Random Forest score
3.   Key XGBoost params: n_estimators, max_depth, learning_rate, subsample, colsample_bytree


### Task 1 - Gradient Boosting intuition
 Random Forest builds trees in parallel and averages them. How does Gradient Boosting builds trees differently?

1.  ## Gradient Boosting vs Random Forest

Random Forest: N trees built independently in parallel on bootstrap samples,
predictions averaged. Reduces variance.

Gradient Boosting: Trees built sequentially. Each new tree trains on the
pseudo-residuals (errors) of the previous prediction, not the original labels.
Each tree's contribution is scaled by a learning rate before adding to the
ensemble. Process repeats until n_estimators trees are built or residuals
stop improving.

Key difference: Random Forest reduces variance by averaging.
Gradient Boosting reduces bias by iteratively correcting errors.

###Task 2 - XGBoost on Titanic
Install, train, compare CV score to your Random Forest (0.814)



In [25]:


import pandas as pd
import numpy as np


df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

df.fillna({'Age': df['Age'].median()}, inplace=True)
df.dropna(subset=['Embarked'], inplace=True)
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.')
df['AgeGroup'] = df['Age'].apply(lambda x: 'Child' if x<13 else 'Teen' if x<18 else 'Adult' if x<61 else 'Senior')
df['Sex_encoded'] = df['Sex'].map({'male':0, 'female':1})

In [26]:
features = ['Pclass', 'Age', 'Fare', 'Sex_encoded']
X = df[features]
y = df['Survived']

In [27]:
X.shape

(889, 4)

In [28]:
import xgboost
print(xgboost.__version__)

3.2.0


In [29]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

#
bst = XGBClassifier()


In [30]:
cv_score = cross_val_score(bst, X,y, cv=5)
print(cv_score)

[0.78651685 0.81460674 0.85955056 0.7752809  0.84180791]


In [31]:
np.mean(cv_score)

np.float64(0.815552593156859)

### XGBoost 0.816 vs Random Forest 0.814
  - essentially the same with default params.

Task 3 - Key params
1. Try adjusting n_estimators, max_depth, learning_rate, subsample, colsample_bytree -> observe what each does to the score


In [32]:
bst1 = XGBClassifier(n_estimators=500, learning_rate=0.01)
#low learning rate + more trees

#CV
cv_score1 = cross_val_score(bst1, X,y, cv=5)
print(cv_score1,"\n")
print(np.mean(cv_score1))

[0.8258427  0.81460674 0.88764045 0.82022472 0.84745763] 

0.8391544467720434


### Best score so far: **0.839** (n_estimators=500, learning_rate=0.01)

In [33]:
#big learning rate + less trees
bst2 = XGBClassifier(n_estimators = 100, learning_rate = 0.3)

#cv
cv_score2 = cross_val_score(bst2, X, y,cv=5)
print(cv_score2, "\n")
print("For 100 trees and 0.3 learning_RATE CV is: ",np.mean(cv_score2))

[0.78651685 0.81460674 0.85955056 0.7752809  0.84180791] 

For 100 trees and 0.3 learning_RATE CV is:  0.815552593156859


In [34]:

#max_depth

#depth =3
bst_m1  = XGBClassifier(max_depth=3)
cv_m1 = cross_val_score(bst_m1,X,y,cv=5)
print("m3", cv_m1)
print(np.mean(cv_m1),"\n")

#depth = 9
bst_m9 = XGBClassifier(max_depth = 9)
cv_m9 = cross_val_score(bst_m9,X,y,cv=5)
print("m9", cv_m9)
print(np.mean(cv_m9))


m3 [0.80337079 0.8258427  0.87078652 0.84269663 0.81355932]
0.8312511902494762 

m9 [0.80337079 0.80337079 0.84269663 0.78089888 0.8079096 ]
0.8076493366342918


###
1. depth=3 → 0.831 (better = simpler trees, less overfit)
2. depth=9 → 0.808 (worse = too deep, memorising noise)

In [35]:
#subsample,
    #fraction of rows randomly sampled per tree (like bootstrap in Random Forest)

bst1 = XGBClassifier(n_estimators=500, learning_rate=0.01,subsample = 0.8, colsample_bytree=0.8)


#CV
cv_score1 = cross_val_score(bst1, X,y, cv=5)
print(cv_score1,"\n")
print(np.mean(cv_score1))

#colsample_bytree
  #colsample_bytree - fraction of features randomly sampled per tree (like max_features in Random Forest)

[0.80898876 0.8258427  0.87640449 0.81460674 0.82485876] 

0.830140290738272


In [36]:
#subsample
subsample_ = [0.6, 0.8, 1.0]

for i in subsample_ :
  bst1 = XGBClassifier(subsample = i)


#CV
  cv_score1 = cross_val_score(bst1, X,y, cv=5)
  print(cv_score1)
  print(np.mean(cv_score1))
  print("\n")

[0.80337079 0.80898876 0.85955056 0.78089888 0.81355932]
0.8132736621595885


[0.79775281 0.78089888 0.83707865 0.78651685 0.80225989]
0.8009014156033771


[0.78651685 0.81460674 0.85955056 0.7752809  0.84180791]
0.815552593156859




In [37]:
#colsample_bytree
cols_sample = [0.6, 0.8, 1.0]

for i in cols_sample:
  bst1 = XGBClassifier(colsample_bytree = i)


#CV
  cv_score1 = cross_val_score(bst1, X,y, cv=5)
  print(cv_score1)
  print(np.mean(cv_score1))
  print("\n")


[0.81460674 0.81460674 0.85955056 0.8258427  0.82485876]
0.8278930997270362


[0.75280899 0.80337079 0.87078652 0.78651685 0.82485876]
0.8076683806259124


[0.78651685 0.81460674 0.85955056 0.7752809  0.84180791]
0.815552593156859




Task 4 : Markdown
One line each explaining what each param controls

## XGBoost Key Parameters

- n_estimators: Number of boosting rounds (trees). More trees with a low
  learning rate generally improves performance up to a point.

- max_depth: Maximum depth of each tree. Lower values reduce overfitting
  (high bias, low variance); higher values risk memorising noise.

- learning_rate: Shrinks each tree's contribution. Lower values require
  more trees but generalise better -> always pair low learning_rate with
  high n_estimators.

- subsample: Fraction of training rows sampled per tree. Adds randomness
  to decorrelate trees and reduce overfitting. Typical range: 0.6–0.9.

- colsample_bytree: Fraction of features sampled per tree. Same purpose
  as subsample but across feature dimension. Similar to max_features
  in Random Forest.

### Best score so far: **0.839** (n_estimators=500, learning_rate=0.01)